# Error Handling

How to handle errors across the Jockey API, including retry strategies, polling helpers, and batch polling for async operations.

In [ ]:
import json
import os
import time
import concurrent.futures

import requests

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
BASE_URL = "https://api.twelvelabs.io/v1.3"
HEADERS = {"x-api-key": API_KEY, "Content-Type": "application/json"}

# Replace with your knowledge store ID
STORE_ID = "your_knowledge_store_id"

## Error Response Format

All errors follow the same shape:

```json
{
  "code": "invalid_request",
  "message": "The 'method' field is required",
  "docs_url": "https://docs.twelvelabs.io/errors/invalid_request"
}
```

## HTTP Status Codes

| Code | Meaning | Action |
|------|---------|--------|
| `200` | Success | Process response |
| `201` | Created | Resource created successfully |
| `202` | Accepted | Async operation started (knowledge store items) |
| `400` | Bad Request | Fix request parameters |
| `401` | Unauthorized | Check API key |
| `403` | Forbidden | Check permissions |
| `404` | Not Found | Check resource ID |
| `429` | Rate Limited | Back off and retry |
| `500` | Server Error | Retry with backoff |

## Retry Strategy

Use exponential backoff for rate-limited (429) and server error (5xx) responses.

In [ ]:
def api_request(
    method: str,
    url: str,
    headers: dict,
    max_retries: int = 3,
    base_delay: int = 1,
    **kwargs,
) -> requests.Response:
    """Make an API request with exponential backoff retry.

    Retries on 429 (rate limited) and 5xx (server error) responses.

    Args:
        method: HTTP method (GET, POST, etc.).
        url: The request URL.
        headers: Request headers.
        max_retries: Maximum number of retry attempts.
        base_delay: Base delay in seconds for exponential backoff.
        **kwargs: Additional arguments passed to requests.request().

    Returns:
        The response object.
    """
    for attempt in range(max_retries):
        response = requests.request(method, url, headers=headers, **kwargs)

        if response.status_code == 429:
            delay = base_delay * (2 ** attempt)
            print(f"Rate limited, retrying in {delay}s...")
            time.sleep(delay)
            continue

        if response.status_code >= 500:
            delay = base_delay * (2 ** attempt)
            print(f"Server error, retrying in {delay}s...")
            time.sleep(delay)
            continue

        return response

    return response  # Return last response after all retries

In [ ]:
# Example: using the retry wrapper
response = api_request(
    "POST",
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": "Summarize these videos",
            }
        ],
        "knowledge_store_id": STORE_ID,
    },
)

print(f"Status: {response.status_code}")
if response.ok:
    print("Success!")
else:
    error = response.json()
    print(f"Error: {error.get('code', 'unknown')} - {error.get('message', '')}")

## Common Errors by Endpoint

### Assets

| Error | Cause | Fix |
|-------|-------|-----|
| `method` required | Missing upload method | Add `method: "direct"` or `method: "url"` |
| File too large | Direct upload > 200MB | Use URL upload method instead |
| Invalid URL | URL not accessible | Check URL is public and reachable |

### Knowledge Stores

| Error | Cause | Fix |
|-------|-------|-----|
| Invalid schema | Bad JSON Schema in ingestion config | Validate against JSON Schema draft 2020-12 |
| Name required | Missing `name` field | Add a name |

### Knowledge Store Items

| Error | Cause | Fix |
|-------|-------|-----|
| Asset not found | Invalid `asset_id` | Check asset exists and is `ready` |
| Store not found | Invalid `knowledge_store_id` | Check knowledge store ID |

### Responses

| Error | Cause | Fix |
|-------|-------|-----|
| Knowledge store required | Missing `knowledge_store_id` | Add `knowledge_store_id` to the request body |
| Invalid session | Bad `session_id` | Start a new session (omit session_id) |
| Input required | Missing `input` array | Add at least one message |

## Polling and Async Status

Jockey processes content asynchronously. Any time you create an asset or add a video to a knowledge store, processing happens in the background. You must poll for completion before proceeding.

### Polling Helper

In [ ]:
def wait_for_ready(
    url: str,
    headers: dict,
    interval: int = 5,
    timeout: int = 600,
) -> dict:
    """Poll a resource until it reaches 'ready' or 'failed' status.

    Args:
        url: The resource URL to poll.
        headers: Request headers.
        interval: Seconds between poll attempts.
        timeout: Maximum seconds to wait before raising TimeoutError.

    Returns:
        The resource dict once it reaches 'ready' status.

    Raises:
        Exception: If the resource reaches 'failed' status.
        TimeoutError: If the resource does not become ready within the timeout.
    """
    elapsed = 0
    while elapsed < timeout:
        response = requests.get(url, headers=headers)
        resource = response.json()
        status = resource["status"]

        if status == "ready":
            return resource
        elif status == "failed":
            raise Exception(f"Resource failed: {resource}")

        print(f"  Status: {status} ({elapsed}s elapsed)")
        time.sleep(interval)
        elapsed += interval

    raise TimeoutError(f"Not ready after {timeout}s")

### Polling Usage

Use appropriate intervals for different resource types.

In [ ]:
# Example asset and item IDs (replace with real values)
ASSET_ID = "your_asset_id"
ITEM_ID = "your_item_id"

# Wait for asset (shorter interval -- uploads finish faster)
# asset = wait_for_ready(f"{BASE_URL}/assets/{ASSET_ID}", HEADERS, interval=5)

# Wait for knowledge store item (longer interval -- indexing takes more time)
# item = wait_for_ready(
#     f"{BASE_URL}/knowledge-stores/{STORE_ID}/items/{ITEM_ID}",
#     HEADERS,
#     interval=10,
#     timeout=600,
# )

print("Uncomment the lines above with real IDs to run polling.")

### Recommended Intervals

| Resource | Poll Interval | Typical Wait | Timeout |
|----------|--------------|-------------|--------|
| Asset (direct) | 5s | 10-60s | 120s |
| Asset (URL) | 5s | 10-120s | 300s |
| Knowledge store item | 10s | 1-10 min | 600s |

### Batch Polling

Wait for multiple resources in parallel using `concurrent.futures`.

In [ ]:
def wait_for_items(
    store_id: str,
    item_ids: list[str],
    max_workers: int = 5,
) -> dict[str, dict]:
    """Wait for multiple knowledge store items in parallel.

    Args:
        store_id: The knowledge store ID.
        item_ids: List of item IDs to wait for.
        max_workers: Maximum number of concurrent polling threads.

    Returns:
        A dict mapping item_id to its ready resource dict.
    """
    def wait_one(item_id: str) -> dict:
        return wait_for_ready(
            f"{BASE_URL}/knowledge-stores/{store_id}/items/{item_id}",
            HEADERS,
            interval=10,
        )

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(wait_one, iid): iid for iid in item_ids}
        results = {}
        for future in concurrent.futures.as_completed(futures):
            item_id = futures[future]
            try:
                results[item_id] = future.result()
                print(f"  Item {item_id}: ready")
            except Exception as exc:
                print(f"  Item {item_id}: failed ({exc})")
                results[item_id] = {"status": "failed", "error": str(exc)}

    return results

In [ ]:
# Example: batch polling (replace with real item IDs)
# item_ids = ["item_001", "item_002", "item_003"]
# results = wait_for_items(STORE_ID, item_ids)
# for item_id, resource in results.items():
#     print(f"{item_id}: {resource['status']}")

print("Uncomment the lines above with real item IDs to run batch polling.")

### Polling Pitfalls

- **Do not poll too aggressively** -- 1-second intervals waste rate limit budget
- **Always handle `failed`** -- failed resources do not recover
- **Set a timeout** -- do not poll forever if something goes wrong
- **Webhooks not available yet** -- polling is the only option in Private Beta

## Next Steps

- [Streaming](./streaming.ipynb) -- Receive responses in real-time via SSE
- [Structured Output](./structured_output.ipynb) -- Force Jockey to return typed JSON
- [Multi-Turn Sessions](./multi_turn_sessions.ipynb) -- Continue conversations across multiple requests
- [API Reference: POST /responses](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/responses/create-response)